In [13]:
import pulp
import copy

#rezolva LP (Linear Programming) relaxat pentru un set de limite
def solve_lp(bounds):
    model = pulp.LpProblem("node", pulp.LpMinimize)

    # variabile continue (valorile nu sunt intregi)
    Xc = pulp.LpVariable("Xc", lowBound=bounds["Xc"][0], upBound=bounds["Xc"][1])
    Xp = pulp.LpVariable("Xp", lowBound=bounds["Xp"][0], upBound=bounds["Xp"][1])
    Xi = pulp.LpVariable("Xi", lowBound=bounds["Xi"][0], upBound=bounds["Xi"][1])
    Xm = pulp.LpVariable("Xm", lowBound=bounds["Xm"][0], upBound=bounds["Xm"][1])

    #functie obiectiv
    model += 3*Xc/100 + 8*Xp/100 + 4*Xi/100 + 2*Xm/100

    #constrangeri
    model += 255/100*Xc + 172/100*Xp + 100/100*Xi + 152/100*Xm >= 2000
    model += 5/100*Xc + 25/100*Xp + 9/100*Xi + 0.8/100*Xm >= 70
    model += 48.3/100*Xc + 0*Xp + 4.7/100*Xi + 33.8/100*Xm >= 40

    #rezolvare LP
    status = model.solve(pulp.PULP_CBC_CMD(msg=0))

    if status != 1:
        #LP infezabil sau fara solutie optima
        return None, None

    sol = {
        "Xc": pulp.value(Xc),
        "Xp": pulp.value(Xp),
        "Xi": pulp.value(Xi),
        "Xm": pulp.value(Xm)
    }

    return pulp.value(model.objective), sol


# verifica daca toate variabilele sunt intregi
def is_integer_solution(sol, eps=1e-6):
    for v in sol.values():
        if abs(v - round(v)) > eps:
            return False
    return True


#algoritmul branch and bound
def branch_and_bound():
    best_value = float("inf")
    best_solution = None

    #limite initiale pentru variabile
    root_bounds = {
        "Xc": (1, 250),
        "Xp": (1, None),
        "Xi": (1, 300),
        "Xm": (1, 300)
    }

    #folosim o stiva pentru explorare
    stack = [root_bounds]

    while stack:
        bounds = stack.pop()

        #rezolvam LP relaxat pentru nodul curent
        obj, sol = solve_lp(bounds)

        if sol is None:
            #nod infezabil
            continue

        #bounding: daca obiectivul e deja mai mare decat cel mai bun gasit, ignoram nodul
        if obj >= best_value:
            continue

        #daca solutia LP este intreaga, actualizam best
        if is_integer_solution(sol):
            if obj < best_value:
                best_value = obj
                best_solution = {k: int(round(v)) for k, v in sol.items()}
            continue

        #alegem prima variabila fractionara pentru branching
        frac_var = None
        for var, val in sol.items():
            if abs(val - round(val)) > 1e-6:
                frac_var = var
                break

        #daca nu exista variabile fractionare, trecem mai departe
        if frac_var is None:
            continue

        val = sol[frac_var]
        floor_val = int(val)
        ceil_val = floor_val + 1

        #cream doua ramuri: var <= floor si var >= ceil
        low_bounds = copy.deepcopy(bounds)
        high_bounds = copy.deepcopy(bounds)

        #actualizam limitele pentru variabila aleasa
        low_bounds[frac_var] = (bounds[frac_var][0], floor_val)
        high_bounds[frac_var] = (ceil_val, bounds[frac_var][1])

        #adaugam ramurile in stiva
        stack.append(low_bounds)
        stack.append(high_bounds)

    return best_solution, best_value


#rulare
if __name__ == "__main__":
    sol, val = branch_and_bound()
    print("solutie branch and bound:")
    print(sol)
    print("valoare minima:", val)

solutie branch and bound:
{'Xc': 250, 'Xp': 353, 'Xi': 300, 'Xm': 300}
valoare minima: 53.74
